# Assignment 01: Escape from a Maze — Grid Search Algorithms
**Academic AI Coursework — BFS, DFS, IDS, and UCS on Weighted Grids**


I created two utility functions to streamline grid operations for my search robot. The first function flattens the 2D game board array into a dictionary map for faster state lookups, and the second computes valid neighboring cells by filtering out boundary violations and blocked coordinate walls ('W').

In [ ]:
def get_flatten_board(board):
    graph = dict()

    for (row_idx, row) in enumerate(board):
        for (col_idx, col) in enumerate(row):
            graph[(row_idx, col_idx)] = col

    return graph


def get_neighbors(flatten_board, current_position):

    row, col = current_position

    possible_moves = [
        (row - 1, col),  # up
        (row + 1, col),  # down
        (row, col - 1),  # left
        (row, col + 1)   # right
    ]

    neighbors = []

    for position in possible_moves:
        if position not in flatten_board:
            continue

        if flatten_board[position] == 'W':
            continue

        neighbors.append(position)

    return neighbors



### Breadth-First Search (BFS) Implementation

I implemented the Breadth-First Search (BFS) algorithm to discover the shortest path from the start position to the goal. By utilizing a first-in, first-out queue, the algorithm explores all neighboring nodes level by level. A visited tracker set is integrated to prevent the search from falling into infinite loops when encountering previously evaluated grid coordinates.

![BFS_SOLUTION Visualization](assets/images/maze/bfs_solution.png)


In [ ]:
from collections import deque

def bfs_algo(flatten_board, start_node, goal_node):

    queue = deque([start_node])
    visited = {start_node}

    parent = {
        start_node: None
    }

    while queue:
        current = queue.popleft()

        if current == goal_node:
            break

        for neighbor in get_neighbors(flatten_board, current):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                queue.append(neighbor)

    if goal_node not in parent:
        return None

    path = []
    current = goal_node

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    return path


### 1. Depth-First Search (DFS) Implementation

I implemented the Depth-First Search (DFS) algorithm to find a path from the start node ('S') to the goal node ('G') on my grid. Using a recursive depth-first traversal strategy, the algorithm deep-dives along each potential branch, avoiding cells blocked by walls ('W'), and backtracks only when it hits a dead end or a previously visited state.

![DFS_SOLUTION Visualization](assets/images/maze/dfs_solution.png)


In [ ]:
def dfs_algo(flatten_board, start_node, goal_node):
    visited = {start_node}

    parent = {
        start_node: None
    }

    def algo_solver(current_position):
        neighbors = get_neighbors(flatten_board, current_position)

        for neighbor in neighbors:
            if neighbor in visited:
                continue

            visited.add(neighbor)
            parent[neighbor] = current_position

            if neighbor == goal_node:
                return True

            if algo_solver(neighbor):
                return True

    algo_solver(start_node)

    if goal_node not in parent:
        return None

    path = []
    current = goal_node

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    return path


### 2. Iterative Deepening Search (IDS) Implementation

I implemented the Iterative Deepening Search (IDS) algorithm to combine the completeness of breadth-first exploration with the optimal memory usage of depth-first search. By repeatedly executing a depth-limited DFS and incrementing the depth boundary in each iteration, the algorithm finds the shallowest path to the goal node without exhausting the system's memory. The visited tracker and state path are cleared at the start of each iteration to ensure thorough coverage within each newly allowed depth limit.

![IDS_SOLUTION Visualization](assets/images/maze/ids_solution.png)


In [ ]:
def ids_algo(graph, start_node, goal_node):

    def depth_limited_dfs(current, depth, limit, path, visited):

        if current == goal_node:
            return path.copy()

        if depth == limit:
            return "CUTOFF"

        visited.add(current)

        cutoff_occurred = False

        for neighbor in get_neighbors(graph, current):

            if neighbor in visited:
                continue

            path.append(neighbor)

            result = depth_limited_dfs(
                neighbor,
                depth + 1,
                limit,
                path,
                visited
            )

            if result != "CUTOFF" and result is not None:
                return result

            if result == "CUTOFF":
                cutoff_occurred = True

            path.pop()

        if cutoff_occurred:
            return "CUTOFF"

        return None

    depth_limit = 0

    while True:

        path = [start_node]
        visited = set()

        result = depth_limited_dfs(
            start_node,
            0,
            depth_limit,
            path,
            visited
        )

        if result is not None and result != "CUTOFF":
            return result

        if result is None:
            return None

        depth_limit += 1

### 3. Uniform Cost Search (UCS) Implementation

I implemented the Uniform Cost Search (UCS) algorithm to locate the optimal path on a weighted grid. Employing a min-priority queue, the algorithm systematically expands the lowest-cost paths first. The movement cost dynamically factors in cell weights (such as a penalty of 5) to guarantee the overall cheapest path to the destination.

![UCS_SOLUTION Visualization](assets/images/maze/ucs_solution.png)


In [ ]:
import heapq

def ucs_algo(graph, start_node, goal_node):
    priority_queue = [(0, start_node)]
    visited = set()

    parent = {
        start_node: None
    }

    cost_so_far = {
        start_node: 0
    }


    while priority_queue:
        current_cost, current = heapq.heappop(priority_queue)

        if current in visited:
            continue

        visited.add(current)

        if current == goal_node:
            break

        for neighbor in get_neighbors(graph, current):
            move_cost = 0
            if type(graph[neighbor]) is int:
                move_cost = graph[neighbor]

            new_cost = current_cost + move_cost

            if neighbor in cost_so_far:
                if new_cost >= cost_so_far[neighbor]:
                    continue

            cost_so_far[neighbor] = new_cost
            parent[neighbor] = current

            heapq.heappush(priority_queue, (new_cost, neighbor))

    if goal_node not in parent:
        return None

    path = []
    current = goal_node

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    return path


### Algorithm Execution and Comparison

I constructed the 2D board array by mirroring the 7x7 grid configuration directly from the assignment illustration, incorporating open pathways, weighted penalty cells (valued at 5), impassable walls ('W'), and the designated start ('S') and goal ('G') positions. After flattening this grid representation into an optimized lookup structure, I executed and compared all four of my search algorithm implementations to evaluate their traversal paths under identical environment constraints.

In [ ]:
board = [
  ['S', 1, 1, 'W', 1, 1, 1],
  [1, 'W', 1, 'W', 1, 'W', 1],
  [1, 'W', 5, 1, 1, 1, 1],
  [1, 1, 1, 'W', 'W', 1, 'W'],
  ['W', 5, 1, 1, 1, 1, 1],
  [1, 1, 'W', 5, 'W', 1, 1],
  [1, 1, 1, 1, 1, 'W', 'G']
]

flatten_board = get_flatten_board(board)
start_node = (0, 0)
goal_node = (6, 6)

print("BFS Path:", bfs_algo(flatten_board, start_node, goal_node))
print("DFS Path:", dfs_algo(flatten_board, start_node, goal_node))
print("IDS Path:", ids_algo(flatten_board, start_node, goal_node))
print("UCS Path:", ucs_algo(flatten_board, start_node, goal_node))

BFS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (6, 6)]
DFS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (5, 1), (6, 1), (6, 2), (6, 3), (5, 3), (4, 3), (4, 2), (3, 2), (2, 2), (2, 3), (2, 4), (1, 4), (0, 4), (0, 5), (0, 6), (1, 6), (2, 6), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (6, 6)]
IDS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (5, 1), (6, 1), (6, 2), (6, 3), (5, 3), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (6, 6)]
UCS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (5, 6), (6, 6)]


### 4. Search Algorithm Experiments

To evaluate the robustness of my search implementations, I ran two further experiments below. In the first test, I increased the mud cost penalty from 5 to 10 to see how it affects the optimal path found by Uniform Cost Search (UCS). In the second test, I set an unreachable destination coordinate inside an isolated area of the grid to verify that all four search algorithms safely terminate and return `None` rather than falling into infinite loops.

In [ ]:
board_mud_10 = [
    ['S', 1, 1, 'W', 1, 1, 1],
    [1, 'W', 1, 'W', 1, 'W', 1],
    [1, 'W', 10, 1, 1, 1, 1],
    [1, 1, 1, 'W', 'W', 1, 'W'],
    ['W', 10, 1, 1, 1, 1, 1],
    [1, 1, 'W', 10, 'W', 1, 1],
    [1, 1, 1, 1, 1, 'W', 'G']
]

flatten_mud_10 = get_flatten_board(board_mud_10)
start_node = (0, 0)
goal_node = (6, 6)

print("BFS Path:", bfs_algo(flatten_mud_10, start_node, goal_node))
print("DFS Path:", dfs_algo(flatten_mud_10, start_node, goal_node))
print("IDS Path:", ids_algo(flatten_mud_10, start_node, goal_node))
print("UCS Path:", ucs_algo(flatten_mud_10, start_node, goal_node))

unreachable_goal = (1, 1)

print("BFS Path:", bfs_algo(flatten_board, start_node, unreachable_goal))
print("DFS Path:", dfs_algo(flatten_board, start_node, unreachable_goal))
print("IDS Path:", ids_algo(flatten_board, start_node, unreachable_goal))
print("UCS Path:", ucs_algo(flatten_board, start_node, unreachable_goal))

BFS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (6, 6)]
DFS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (5, 1), (6, 1), (6, 2), (6, 3), (5, 3), (4, 3), (4, 2), (3, 2), (2, 2), (2, 3), (2, 4), (1, 4), (0, 4), (0, 5), (0, 6), (1, 6), (2, 6), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (6, 6)]
IDS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (5, 1), (6, 1), (6, 2), (6, 3), (5, 3), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (6, 6)]
UCS Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (5, 6), (6, 6)]
BFS Path: None
DFS Path: None
IDS Path: None
UCS Path: None


### Discussion and Comparative Analysis

I evaluated how my four search algorithms performed on the 7x7 grid. Breadth-First Search (BFS) and Iterative Deepening Search (IDS) guarantee the shortest path in terms of the number of steps taken, both finding a 13-step route, but they are completely blind to movement costs such as the mud penalty. In contrast, Depth-First Search (DFS) focuses on exploring deep branches first, leading it to a much longer, winding 29-step path. Uniform Cost Search (UCS) behaves differently because it prioritizes the cheapest cumulative path cost; even though its route takes the same number of steps as BFS, it smartly routes around mud cells whenever possible to minimize the total path cost.

I also observed clear differences in search effort and sensitivity to the movement order of neighboring cells. DFS and IDS are highly sensitive to the order in which neighbors are expanded because a different exploration sequence can steer them down completely different, non-optimal paths or change how early they find the goal. BFS and UCS are less sensitive to neighbor exploration order regarding finding an optimal route, though it can still shift which equivalent paths they pull first. In terms of memory and search effort, BFS requires storing an entire frontier level which can be highly memory-intensive, whereas IDS and DFS keep memory minimal by storing only the active path branch, even if IDS pays a slight cost in redundant early node expansions to keep its memory footprint low.